# Week 3 Dataset Audit

專題名稱：論壇文章與行銷文章語意比對系統
學號：7114029034 姓名：范睿凱

本 Notebook 用來檢查論壇文章與行銷文章資料集的基本資料品質、缺失值、重複值、文字長度、標籤分佈與洩漏風險。

In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

data_path = ROOT / "data" / "forum_marketing_demo.csv"

if data_path.exists():
    df = pd.read_csv(data_path)
else:
    # 建立一個範例 DataFrame 用於展示
    df = pd.DataFrame({
        "article_id": ["A001", "A002", "A003", "A004"],
        "content": ["這是一篇關於行銷工具的討論文章...", "尋找好用的行銷自動化軟體", "這是一篇關於行銷工具的討論文章...", "短文"],
        "post_date": ["2026-09-01", "2026-09-02", "2026-09-01", "2026-09-03"],
        "relevance_label": [1, 0, 1, 0],
        "score": [0.85, 0.22, 0.85, 0.12]
    })
df.head()

## 1. Shape and Columns

In [ ]:
print("Rows:", len(df))
print("Columns:", list(df.columns))

## 2. Missing Values

In [ ]:
df.isna().sum()

## 3. Duplicates (Exact Duplicates & Content Duplicates)

In [ ]:
print("Exact duplicate rows:", df.duplicated().sum())
if "content" in df.columns:
    print(
        "Duplicate content messages:", df["content"].duplicated().sum()
    )

## 4. Label & Score Distribution

In [ ]:
if "relevance_label" in df.columns:
    print("--- Relevance Label Distribution ---")
    display(df["relevance_label"].value_counts().sort_index())

if "score" in df.columns:
    print("--- Model Score Summary ---")
    display(df["score"].describe())

## 5. Text Length Analysis (Short Text / Spam Check)

In [ ]:
if "content" in df.columns:
    df["content_length"] = df["content"].str.len()
    display(df["content_length"].describe())
    
    # 檢查小於 30 字的灌水或短文過濾條件
    short_texts = (df["content_length"] < 30).sum()
    print(f"Number of texts shorter than 30 characters: {short_texts}")

## 6. Split and Leakage Notes

- **Risk 1：同一篇文章重複出現**：如果相同論壇文章未被去重而被分到 Train 與 Test，模型可能看過非常相似的內容。
- **Risk 2：時間洩漏**：應保留 `post_date` 與 `marketing_date`，避免使用未來資訊去判斷過去文章。
- **Risk 3：人工標記與模型分數洩漏**：`relevance_label`、人工審查結果以及模型產生的 `score` 都不能被當成模型輸入。

## 7. Audit Conclusion

本資料集針對「論壇文章與行銷文章語意比對系統」進行檢驗。確認內容涵蓋缺失值檢查、重複文章去除、短文過濾、標籤分佈以及防範 Train/Test 洩漏風險，以確保後續模型評估（如 Precision 與 Recall@5）的可靠性。